# Variance Reduction - Weight Windows

## Creating and utilizing a weight window to accelerate deep shielding simulations

This example simulates a shield room / bunker with corridor entrance and a neutron source in the center of the room. This example implements a single step of the Magic method of weight window generation. 

In this tutorial we shall focus on generating a weight window to accelerate the simulation of particles through a shield.

Weight Windows are found using the MAGIC method and used to accelerate the simulation.

The variance reduction method used for this simulation is well documented in the OpenMC documentation
https://docs.openmc.org/en/stable/methods/neutron_physics.html

The MAGIC method is well described in the original publication
https://scientific-publications.ukaea.uk/wp-content/uploads/Published/INTERN1.pdf


First we import ```openmc``` and other packages needed for the example and configure the nuclear data path

In [ ]:
import time  # used to time the simulation
import numpy as np

from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm  # used for plotting log scale graphs

import openmc
# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = '/nuclear_data/cross_sections.xml'
openmc.config['cross_sections'] = '/home/jon/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'


We create a couple of materials for the simulation

In [2]:
mat_air = openmc.Material(name="air")
mat_air.add_element("N", 0.784431)
mat_air.add_element("O", 0.210748)
mat_air.add_element("Ar", 0.0046)
mat_air.set_density("g/cc", 0.001205)

mat_concrete = openmc.Material(name='concrete')
mat_concrete.add_element("H",0.168759)
mat_concrete.add_element("C",0.001416)
mat_concrete.add_element("O",0.562524)
mat_concrete.add_element("Na",0.011838)
mat_concrete.add_element("Mg",0.0014)
mat_concrete.add_element("Al",0.021354)
mat_concrete.add_element("Si",0.204115)
mat_concrete.add_element("K",0.005656)
mat_concrete.add_element("Ca",0.018674)
mat_concrete.add_element("Fe",0.00426)
mat_concrete.set_density("g/cm3", 2.3)

materials_continuous_xs = openmc.Materials([mat_air, mat_concrete])

Now we define and plot the geometry. This geometry is defined by parameters for every width and height. The parameters input into the geometry in a stacked manner so they can easily be adjusted to change the geometry without creating overlapping cells.

In [3]:
width_a = 100
width_b = 100
width_c = 500
width_d = 100
width_e = 100
width_f = 100
width_g = 100

depth_a = 100
depth_b = 100
depth_c = 700
depth_d = 600
depth_e = 100
depth_f = 100

height_j = 100
height_k = 500
height_l = 100

xplane_0 = openmc.XPlane(x0=0, boundary_type="vacuum")
xplane_1 = openmc.XPlane(x0=xplane_0.x0 + width_a)
xplane_2 = openmc.XPlane(x0=xplane_1.x0 + width_b)
xplane_3 = openmc.XPlane(x0=xplane_2.x0 + width_c)
xplane_4 = openmc.XPlane(x0=xplane_3.x0 + width_d)
xplane_5 = openmc.XPlane(x0=xplane_4.x0 + width_e)
xplane_6 = openmc.XPlane(x0=xplane_5.x0 + width_f)
xplane_7 = openmc.XPlane(x0=xplane_6.x0 + width_g, boundary_type="vacuum")

yplane_0 = openmc.YPlane(y0=0, boundary_type="vacuum")
yplane_1 = openmc.YPlane(y0=yplane_0.y0 + depth_a)
yplane_2 = openmc.YPlane(y0=yplane_1.y0 + depth_b)
yplane_3 = openmc.YPlane(y0=yplane_2.y0 + depth_c)
yplane_4 = openmc.YPlane(y0=yplane_3.y0 + depth_d)
yplane_5 = openmc.YPlane(y0=yplane_4.y0 + depth_e)
yplane_6 = openmc.YPlane(y0=yplane_5.y0 + depth_f, boundary_type="vacuum")

zplane_1 = openmc.ZPlane(z0=0, boundary_type="vacuum")
zplane_2 = openmc.ZPlane(z0=zplane_1.z0 + height_j)
zplane_3 = openmc.ZPlane(z0=zplane_2.z0 + height_k)
zplane_4 = openmc.ZPlane(z0=zplane_3.z0 + height_l, boundary_type="vacuum")

outside_left_region = +xplane_0 & -xplane_1 & +yplane_1 & -yplane_5 & +zplane_1 & -zplane_4
wall_left_region = +xplane_1 & -xplane_2 & +yplane_2 & -yplane_4 & +zplane_2 & -zplane_3
wall_right_region = +xplane_5 & -xplane_6 & +yplane_2 & -yplane_5 & +zplane_2 & -zplane_3
wall_top_region = +xplane_1 & -xplane_4 & +yplane_4 & -yplane_5 & +zplane_2 & -zplane_3
outside_top_region = +xplane_0 & -xplane_7 & +yplane_5 & -yplane_6 & +zplane_1 & -zplane_4
wall_bottom_region = +xplane_1 & -xplane_6 & +yplane_1 & -yplane_2 & +zplane_2 & -zplane_3
outside_bottom_region = +xplane_0 & -xplane_7 & +yplane_0 & -yplane_1 & +zplane_1 & -zplane_4
wall_middle_region = +xplane_3 & -xplane_4 & +yplane_3 & -yplane_4 & +zplane_2 & -zplane_3
outside_right_region = +xplane_6 & -xplane_7 & +yplane_1 & -yplane_5 & +zplane_1 & -zplane_4

room_region = +xplane_2 & -xplane_3 & +yplane_2 & -yplane_4 & +zplane_2 & -zplane_3
gap_region = +xplane_3 & -xplane_4 & +yplane_2 & -yplane_3 & +zplane_2 & -zplane_3
corridor_region = +xplane_4 & -xplane_5 & +yplane_2 & -yplane_5 & +zplane_2 & -zplane_3

roof_region = +xplane_1 & -xplane_6 & +yplane_1 & -yplane_5 & +zplane_1 & -zplane_2
floor_region = +xplane_1 & -xplane_6 & +yplane_1 & -yplane_5 & +zplane_3 & -zplane_4

outside_left_cell = openmc.Cell(region=outside_left_region, fill=mat_air)
outside_right_cell = openmc.Cell(region=outside_right_region, fill=mat_air)
outside_top_cell = openmc.Cell(region=outside_top_region, fill=mat_air)
outside_bottom_cell = openmc.Cell(region=outside_bottom_region, fill=mat_air)
wall_left_cell = openmc.Cell(region=wall_left_region, fill=mat_concrete)
wall_right_cell = openmc.Cell(region=wall_right_region, fill=mat_concrete)
wall_top_cell = openmc.Cell(region=wall_top_region, fill=mat_concrete)
wall_bottom_cell = openmc.Cell(region=wall_bottom_region, fill=mat_concrete)
wall_middle_cell = openmc.Cell(region=wall_middle_region, fill=mat_concrete)
room_cell = openmc.Cell(region=room_region, fill=mat_air)
gap_cell = openmc.Cell(region=gap_region, fill=mat_air)
corridor_cell = openmc.Cell(region=corridor_region, fill=mat_air)

roof_cell = openmc.Cell(region=roof_region, fill=mat_concrete)
floor_cell = openmc.Cell(region=floor_region, fill=mat_concrete)

geometry = openmc.Geometry(
    [
        outside_bottom_cell,
        outside_top_cell,
        outside_left_cell,
        outside_right_cell,
        wall_left_cell,
        wall_right_cell,
        wall_top_cell,
        wall_bottom_cell,
        wall_middle_cell,
        room_cell,
        gap_cell,
        corridor_cell,
        roof_cell,
        floor_cell,
    ]
)

Now we plot the geometry and color by materials.

In [ ]:
plot = geometry.plot(basis='xy',  color_by='material')
plot.figure.savefig('geometry_top_down_view.png', bbox_inches="tight")

Next we create a point source, this also uses the same geometry parameters to place in the center of the room regardless of the values of the parameters.

In [5]:
# location of the point source
source_x = width_a + width_b + width_c * 0.5
source_y = depth_a + depth_b + depth_c * 0.75
source_z = height_j + height_k * 0.5
space = openmc.stats.Point((source_x, source_y, source_z))


# all (100%) of source particles are 2.5MeV energy


source = openmc.IndependentSource(
    space=space,
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([2.5e6], [1.0]),
    particle="neutron"
)

In [6]:
# import openmc.mgxs

mgxs_lib = openmc.mgxs.Library(geometry)

groups = openmc.mgxs.EnergyGroups(openmc.mgxs.GROUP_STRUCTURES['CASMO-2'])  # https://docs.openmc.org/en/latest/pythonapi/mgxs.html#openmc.mgxs.GROUP_STRUCTURES
mgxs_lib.energy_groups = groups
# Disable transport correction
mgxs_lib.correction = None

# Specify needed cross sections for random ray, fixed source does not actually make use of all of these but openmc currently expects to find them so we add them to avoid an error. 
mgxs_lib.mgxs_types = [
    'total', 'absorption', 'nu-fission', 'fission',
    'nu-scatter matrix', 'multiplicity matrix', 'chi'
]

# Specify a "cell" domain type for the cross section tally filters
mgxs_lib.domain_type = "material"

# Specify the cell domains over which to compute multi-group cross sections
mgxs_lib.domains = geometry.get_all_materials().values()

# Do not compute cross sections on a nuclide-by-nuclide basis
mgxs_lib.by_nuclide = False

# Check the library - if no errors are raised, then the library is satisfactory.
mgxs_lib.check_library_for_openmc_mgxs()

# Construct all tallies needed for the multi-group cross section library
mgxs_lib.build_library()

# Create a "tallies.xml" file for the MGXS Library
tallies = openmc.Tallies()
mgxs_lib.add_to_tallies_file(tallies, merge=True)


settings = openmc.Settings()

settings.run_mode = "fixed source"
settings.source = source
settings.particles = 80
settings.batches = 5

model = openmc.Model(geometry, materials_continuous_xs, settings, tallies)




Now we run the simulation to generate a statepoint that can be used to make multigroup cross sections

In [ ]:
!rm s*.h5
mg_statepoint = model.run()

Now we open up the statepoint and summary files to generate the multi group cross sections

In [8]:
summary = openmc.Summary('summary.h5')
geom = summary.geometry
# mats = summary.materials
sp = openmc.StatePoint(mg_statepoint)

groups = openmc.mgxs.EnergyGroups(openmc.mgxs.GROUP_STRUCTURES['CASMO-2'])
mgxs_lib = openmc.mgxs.Library(geom)
mgxs_lib.energy_groups = groups
mgxs_lib.correction = None
mgxs_lib.mgxs_types = ['total', 'absorption', 'nu-fission', 'fission',
                        'nu-scatter matrix', 'multiplicity matrix', 'chi']

# Specify a "cell" domain type for the cross section tally filters
mgxs_lib.domain_type = "material"

# Specify the cell domains over which to compute multi-group cross sections
mgxs_lib.domains = geom.get_all_materials().values()

# Do not compute cross sections on a nuclide-by-nuclide basis
mgxs_lib.by_nuclide = False

# Check the library - if no errors are raised, then the library is satisfactory.
mgxs_lib.check_library_for_openmc_mgxs()

# Construct all tallies needed for the multi-group cross section library
mgxs_lib.build_library()

mgxs_lib.load_from_statepoint(sp)

# assumes that the material names are unique
names = [mat.name for mat in mgxs_lib.domains]

# Create a MGXS File which can then be written to disk
mgxs_file = mgxs_lib.create_mg_library(xs_type='macro', xsdata_names=names)

# Write the file to disk using the default filename of "mgxs.h5"
mgxs_file.export_to_hdf5("mgxs.h5")
openmc.config['mg_cross_sections'] = 'mgxs.h5'

Next we create a mesh that encompasses the entire geometry and scores neutron flux

In [ ]:
mesh = openmc.RegularMesh().from_domain(geometry)
mesh.dimension = (100, 100, 1)
mesh.id = 1

mesh_filter = openmc.MeshFilter(mesh, filter_id=1)
particle_filter = openmc.ParticleFilter('neutron', filter_id=2)

flux_tally = openmc.Tally(name="flux tally")
flux_tally.filters = [mesh_filter, particle_filter]
flux_tally.scores = ["flux"]
flux_tally.id = 42  # we set the ID because we need to access this later

# changing the tallies as previous they were setup for MGXS generation
tallies = openmc.Tallies([flux_tally])

Creates the simulation settings with FW-CADIS weight window generation and getting the flux tally at the same time

In [ ]:
import numpy as np



# Instantiate some Macroscopic Data
air_data = openmc.Macroscopic('air')
concrete_data = openmc.Macroscopic('concrete')

# Instantiate some Materials and register the appropriate Macroscopic objects
air= openmc.Material(name='air')
air.set_density('macro', 1.0)
air.add_macroscopic(air_data)

concrete= openmc.Material(name='concrete')
concrete.set_density('macro', 1.0)
concrete.add_macroscopic(concrete_data)

# Instantiate a Materials collection and export to XML
materials_multi_group_xs = openmc.Materials([air, concrete])
materials_multi_group_xs.cross_sections = "mgxs.h5"

for id, cell in geometry.get_all_cells().items():
    if cell.fill.name == 'air':
        cell.fill = air
    elif cell.fill.name == 'concrete':
        cell.fill = concrete


settings = openmc.Settings()


source = openmc.IndependentSource(
    space=space,
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([2.5e6], [1.0]),
    particle="neutron",
constraints={'domains': [room_cell]}
)

settings.run_mode = "fixed source"
settings.source = source
settings.particles = 80000
settings.batches = 5
# no need to write the tallies.out file which saves space and time when large meshes are used
settings.output = {'tallies': False}
settings.random_ray['adjoint'] = True
settings.energy_mode = 'multi-group'
settings.random_ray['distance_inactive'] = np.sqrt(np.sum(geometry.bounding_box.width))
settings.random_ray['distance_active'] = np.sqrt(np.sum(geometry.bounding_box.width)) * 2

pitch = 1.26
lower_left  = (-pitch, -pitch, -pitch)
upper_right = ( pitch,  pitch,  pitch)
uniform_dist = openmc.stats.Box(lower_left, upper_right)
settings.random_ray['ray_source'] = openmc.IndependentSource(space=uniform_dist)

settings.max_history_splits = 1_000  # controls the maximum partile splits over the entire lifetime of the particle

settings.weight_window_generators = openmc.WeightWindowGenerator(
    mesh=mesh,  # this is the mesh that covers the geometry
    energy_bounds=np.linspace(0.0, 2.5e6, 1),  # 1 energy bins from 0 to max source energy
    particle_type='neutron',
    method='fw_cadis',  # "magic" is the default method so this change is necessary when using FW-CADIS
    max_realizations=settings.batches
)


Now we run the random ray simulation to generate weight windows

In [ ]:
! rm s*.h5
random_ray_wwg_model = openmc.Model(geometry, materials_multi_group_xs, settings, tallies)
random_ray_wwg_statepoint = random_ray_wwg_model.run()

now we should see a weight_windows.h5 file has been created

In [ ]:
!ls -lh weight_windows.h5

now we plot the weight windows just to check

We are going to plot the mesh results with and without weight windows so lets write a function for the plotting

In [ ]:
weight_windows = openmc.hdf5_to_wws('weight_windows.h5')
plt.imshow(
        weight_windows[0].lower_ww_bounds,
        origin='lower', norm=LogNorm())
plt.title('lower_ww_bounds')
plt.colorbar()

In [ ]:
settings = openmc.Settings()

settings.weight_window_checkpoints = {'collision': True, 'surface': True}
settings.survival_biasing = False

settings.weight_windows = weight_windows
settings.weight_windows_on = True

source = openmc.IndependentSource(
    space=space,
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([2.5e6], [1.0]),
    particle="neutron",
)


settings.run_mode = "fixed source"
settings.source = source
settings.particles = 80000
settings.batches = 5

for id, cell in geometry.get_all_cells().items():
    if cell.fill.name == 'air':
        cell.fill = mat_air
    elif cell.fill.name == 'concrete':
        cell.fill = mat_concrete


!rm s*.h5
simulation_using_ww = openmc.Model(geometry, materials_continuous_xs, settings, tallies)
statepoint_with_ww_on = simulation_using_ww.run()

In [25]:
def plot_mesh_tally(statepoint_filename, image_filename):

    with openmc.StatePoint(statepoint_filename) as sp:
        flux_tally = sp.get_tally(name="flux tally")

    mesh_extent = mesh.bounding_box.extent['xy']

    # get a slice of mean values on the xy basis mid z axis
    flux_mean = flux_tally.get_reshaped_data(value='mean', expand_dims=True).squeeze()
    # create a plot of the mean flux values
    plt.imshow(
        flux_mean[:,:,25].T,
        origin="lower",
        extent=mesh_extent,
        norm=LogNorm(),
    )
    plt.title("Flux Mean")

    plt.savefig(image_filename)

In [ ]:
plot_mesh_tally(statepoint_with_ww_on, "flux_mean.png")